In [1]:
import numpy as np 
import pandas as pd 
import kit as erk 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
%matplotlib inline
%load_ext autoreload 
%autoreload 2

In [3]:
n_scenarios = 5000
rates, zc_prices = erk.cir(10, n_scenarios= n_scenarios, b=.03, r_0 = .03, sigma = .02)
prices_eq = erk.gbm(n_years=10, n_scenarios= n_scenarios, mu=.07, sigma=.15)

In [6]:
rets_eq = prices_eq.pct_change().dropna()
rets_zc = zc_prices.pct_change().dropna()

rets_70_30 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.fixedmix_allocator, w1=0.7)
pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor=0.75),
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_70_30, name="70/30", floor=0.75)], axis = 1).round(2)

,ZC,Eq,70/30
mean,1.34,1.98,1.76
stdev,0.00,0.99,0.60
min,1.34,0.37,0.56
max,1.34,9.87,5.54
p_breach,0.00,0.03,0.01
e_short,0.00,0.11,0.07
e_surplus,0.00,0.00,0.00
p_reach,0.00,0.00,0.00


In [7]:
rets_floor75 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, floor=.75, zc_prices = zc_prices[1:])

pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor =0.75),
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_70_30, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75", floor=0.75)],
           axis = 1).round(2)

,ZC,Eq,70/30,Floor75
mean,1.34,1.98,1.76,1.96
stdev,0.00,0.99,0.60,0.99
min,1.34,0.37,0.56,0.76
max,1.34,9.87,5.54,9.87
p_breach,0.00,0.03,0.01,0.00
e_short,0.00,0.11,0.07,0.00
e_surplus,0.00,0.00,0.00,0.00
p_reach,0.00,0.00,0.00,0.00


In [13]:
rets_floor75m1 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, floor=.75, zc_prices = zc_prices[1:], m=1)
rets_floor75m5 = erk.bt_mix(rets_eq, rets_zc, allocator=erk.floor_allocator, floor=.75, zc_prices = zc_prices[1:], m=5)

pd.concat([erk.terminal_stats(rets_zc, name="ZC", floor =0.75),
           erk.terminal_stats(rets_eq, name="Eq", floor=0.75),
           erk.terminal_stats(rets_70_30, name="70/30", floor=0.75),
           erk.terminal_stats(rets_floor75, name="Floor75m", floor=0.75),
           erk.terminal_stats(rets_floor75m1, name="Floor75m1", floor=0.75),
           erk.terminal_stats(rets_floor75m5, name="Floor75m5", floor=0.75)],
           axis = 1).round(4)

,ZC,Eq,70/30,Floor75m,Floor75m1,Floor75m5
mean,1.3433,1.9805,1.7635,1.9568,1.6266,1.9668
stdev,0.0000,0.9880,0.5993,0.9940,0.4360,0.9965
min,1.3433,0.3718,0.5586,0.7605,0.9164,0.7500
max,1.3433,9.8668,5.5388,9.8668,5.0960,9.8668
p_breach,0.0000,0.0342,0.0066,0.0000,0.0000,0.0002
e_short,0.0000,0.1138,0.0718,0.0000,0.0000,0.0000
e_surplus,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
p_reach,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## Risk Budgeting with Drawdown constraints

In [ ]:
cashrate = 0.02
monthly_cashreturn = (1+ cashrate)**(1/12) - 1

rets_cash = pd.DataFrame(data=monthly_cashreturn, index=rets_eq.index, columns = rets_eq.columns)
rets_maxdd25 = erk.bt_mix(rets_eq, rets_cash, allocator=erk.drawdown_allocator, maxdd=.25)
tv_maxdd25 = erk.terminal_stats(rets_maxdd25)